# single file loader

In [1]:
from pathlib import Path
from typing import List

from src.models.document import Document, DocumentMetadata


def load_markdown(file_path: Path) -> List[Document]:
    """
    Load a single markdown file.

    Returns:
        List[Document]
    """

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"Markdown file not found: {file_path}"
        )

    content = file_path.read_text(
        encoding="utf-8"
    )

    metadata = DocumentMetadata(
        source=file_path.name,
        file_type="md",
        page=None,
    )

    return [
        Document(
            page_content=content,
            metadata=metadata
        )
    ]

In [2]:
markdown_docs = load_markdown("../documents/markdown/NemoClaw Overview.md")

In [4]:
markdown_docs

[Document(page_content='> For clean Markdown of any page, append .md to the page URL.\n> For a complete documentation index, see https://docs.nvidia.com/nemoclaw/llms.txt.\n> For full documentation content, see https://docs.nvidia.com/nemoclaw/llms-full.txt.\n> For AI client integration (Claude Code, Cursor, etc.), connect to the MCP server at https://docs.nvidia.com/nemoclaw/_mcp/server.\n\n# Overview of NVIDIA NemoClaw\n\n> NemoClaw is an open-source reference stack that simplifies running OpenClaw always-on assistants more safely.\n\nNVIDIA NemoClaw is an open-source reference stack that simplifies running [OpenClaw](https://openclaw.ai) always-on assistants more safely.\nNemoClaw provides onboarding, lifecycle management, and OpenClaw operations within OpenShell containers.\nIt incorporates policy-based privacy and security guardrails, giving you control over your agents’ behavior and data handling.\nThis enables self-evolving claws to run more safely in clouds, on prem, RTX PCs an

# Directory loader

In [4]:
from pathlib import Path
from typing import List

from src.models.document import Document


def load_markdowns(directory_path: Path) -> List[Document]:
    """
    Load all markdown files from a directory.
    """

    directory_path = Path(directory_path)

    if not directory_path.exists():
        raise FileNotFoundError(
            f"Directory not found: {directory_path}"
        )

    markdown_files = sorted(
        directory_path.glob("*.md")
    )

    documents: List[Document] = []

    for md_file in markdown_files:
        content = md_file.read_text(
            encoding = "utf-8"
        )

        metadata = DocumentMetadata(
            source=md_file.name,
            file_type="md",
            page=None,
        )

        documents.append(
            Document(
                page_content=content,
                metadata=metadata
            )
        )
        
    return documents

In [5]:
markdown_docs = load_markdowns("../documents/markdown")

In [6]:
markdown_docs

[Document(page_content='> For clean Markdown of any page, append .md to the page URL.\n> For a complete documentation index, see https://docs.nvidia.com/nemoclaw/llms.txt.\n> For full documentation content, see https://docs.nvidia.com/nemoclaw/llms-full.txt.\n> For AI client integration (Claude Code, Cursor, etc.), connect to the MCP server at https://docs.nvidia.com/nemoclaw/_mcp/server.\n\n# NemoClaw Architecture Overview\n\n> Learn how NemoClaw combines a host CLI, sandbox plugin, and versioned blueprint to move OpenClaw into a controlled sandbox.\n\nThis page explains how NemoClaw runs OpenClaw inside an OpenShell sandbox and how the gateway connects the agent to inference, integrations, and policy.\n\nNemoClaw does not replace OpenClaw or OpenShell.\nIt packages them into a repeatable setup with a host CLI, a versioned blueprint, default policies, inference setup, plugin configuration, and state helpers.\nYou can use that setup directly or adapt it for your own OpenShell integrati

# Cleaner.py

In [ ]:
import re
from typing import List

from src.models.document import Document


class MarkdownCleaner:

    # Four lines of boilerplate on every markdown file
    BOILERPLATE_PATTERN = re.compile(
        r"^(?:> For clean Markdown.*\n"
        r"> For a complete documentation index.*\n"
        r"> For full documentation content.*\n"
        r"> For AI client integration.*\n?)",
        flags=re.MULTILINE,
    )

    IMAGE_PATTERN = re.compile(          # ![NemoClaw High-Level Component Diagram](url) --> NemoClaw High-Level Component Diagram
    r"!\[(.*?)\]\((.*?)\)"
)
    LINK_PATTERN = re.compile(           # [Architecture](/reference/architecture) --> Architecture
    r"\[([^\]]+)\]\([^)]+\)"
)
    
    BROKEN_LINK_PATTERN = re.compile(
    r"\[([^\]\n]+)"
)
    
    DEFINITION_LIST_PATTERN = re.compile(
    r"^([^\n]+)\n:\s*(.+)$",
    flags=re.MULTILINE,
)
    BLOCKQUOTE_PATTERN = re.compile(
    r"^\s*>\s?",
    re.MULTILINE,
)
    REMOVE_SECTION_PATTERN = re.compile(
    r"^##\s+(?:Next Steps|Related topics)\s*$.*?(?=^##\s+|\Z)",
    flags=re.MULTILINE | re.DOTALL,
)

    MERMAID_PATTERN = re.compile(
        r"```mermaid.*?```",
        flags=re.DOTALL,
    )

    EXCESS_NEWLINES_PATTERN = re.compile(r"\n{3,}")



    def remove_boilerplate(self, text: str) -> str:
        return self.BOILERPLATE_PATTERN.sub("", text)
    
    def remove_image_url(self, text: str) -> str:
        text = self.IMAGE_PATTERN.sub(r"\1", text)
        return text
    
    def remove_links(self, text: str) -> str:
        text = self.LINK_PATTERN.sub(r"\1", text)
        return text
    
    def remove_broken_link_brackets(self, text: str) -> str:
        return self.BROKEN_LINK_PATTERN.sub(r"\1", text)

    def normalize_definition_lists(self, text: str) -> str:
        return self.DEFINITION_LIST_PATTERN.sub(
            r"\1: \2",
            text,
    )

    def remove_blockquote_markers(self, text: str) -> str:
        return self.BLOCKQUOTE_PATTERN.sub("", text)
    
    def remove_navigation_sections(self, text: str) -> str:
        return self.REMOVE_SECTION_PATTERN.sub("", text)

    def remove_mermaid_blocks(self, text: str) -> str:
        return self.MERMAID_PATTERN.sub("", text)

    def normalize_whitespace(self, text: str) -> str:
        text = self.EXCESS_NEWLINES_PATTERN.sub("\n\n", text)
        return text.strip()

    def clean_text(self, text: str) -> str:
        text = self.remove_boilerplate(text)
        text = self.remove_navigation_sections(text)
        text = self.remove_mermaid_blocks(text)
        text = self.remove_image_url(text)
        text = self.remove_links(text)
        text = self.remove_broken_link_brackets(text)
        text = self.normalize_definition_lists(text)
        text = self.remove_blockquote_markers(text)
        text = self.normalize_whitespace(text)
        return text

    def clean_documents(
        self,
        documents: List[Document]
    ) -> List[Document]:

        cleaned_documents = []

        for document in documents:

            cleaned_content = self.clean_text(
                document.page_content
            )

            cleaned_documents.append(
                Document(
                    page_content=cleaned_content,
                    metadata=document.metadata
                )
            )

        return cleaned_documents

In [36]:
cleaner = MarkdownCleaner()
cleaned_markdown_docs = cleaner.clean_documents(markdown_docs)

In [39]:
cleaned_markdown_docs[:3]

[Document(page_content='# NemoClaw Architecture Overview\nLearn how NemoClaw combines a host CLI, sandbox plugin, and versioned blueprint to move OpenClaw into a controlled sandbox.\n\nThis page explains how NemoClaw runs OpenClaw inside an OpenShell sandbox and how the gateway connects the agent to inference, integrations, and policy.\n\nNemoClaw does not replace OpenClaw or OpenShell.\nIt packages them into a repeatable setup with a host CLI, a versioned blueprint, default policies, inference setup, plugin configuration, and state helpers.\nYou can use that setup directly or adapt it for your own OpenShell integration.\n\n## High-Level Flow\n\nNemoClaw keeps the user workflow on the host while OpenShell enforces the sandbox boundary.\nThe gateway sits between NemoClaw control, the sandbox, inference providers, and external integrations.\nThat placement lets NemoClaw configure the environment without giving the agent direct access to host credentials or uncontrolled network egress.\n\

In [41]:
with open("cleaner_output.txt", "w", encoding="utf-8") as f:
    for doc in cleaned_markdown_docs: 
        f.write(f"{doc}\n")

# New Cleaner

In [42]:
import re
from typing import List
from src.models.document import Document

class MarkdownCleaner:
    def __init__(self):
        # 1. Matches exactly the 4 lines of Fern boilerplate at the start of files
        self.BOILERPLATE_PATTERN = re.compile(
            r"^>\s*For clean Markdown.*\n"
            r"^>\s*For a complete documentation index.*\n"
            r"^>\s*For full documentation content.*\n"
            r"^>\s*For AI client integration.*\n",
            flags=re.MULTILINE
        )

        # 2. Removes specific navigation/footer sections cleanly without over-greedy matching
        # Matches from the header line up to the next heading line or true end of file
        self.NAVIGATION_SECTION_PATTERN = re.compile(
            r"^##\s+(?:Next Steps|Related topics)\s*\n(?:(?!^##\s+).)*",
            flags=re.MULTILINE | re.DOTALL
        )

        # 3. Removes Mermaid code blocks cleanly
        self.MERMAID_PATTERN = re.compile(
            r"```mermaid\n.*?```\n?",
            flags=re.DOTALL
        )

        # 4. Extract caption from image tag: ![Caption](url) -> Caption
        self.IMAGE_PATTERN = re.compile(r"!\[([^\]]*)]\([^\)]+\)")

        # 5. Extract anchor text from markdown links: [Anchor](url) -> Anchor
        # Ignores images due to step 4 processing order
        self.LINK_PATTERN = re.compile(r"\[([^\]]+)]\([^\)]+\)")

        # 6. Normalizes definition lists (e.g., Term\n: Definition -> Term: Definition)
        self.DEFINITION_LIST_PATTERN = re.compile(
            r"^([^\n]+)\n:\s*(.+)$",
            flags=re.MULTILINE
        )

        # 7. Removes leading blockquote markers but keeps content context
        self.BLOCKQUOTE_PATTERN = re.compile(r"^\s*>\s*", flags=re.MULTILINE)

        # 8. Cleans excess vertical whitespace while protecting single structural breaks
        self.EXCESS_NEWLINES_PATTERN = re.compile(r"\n{3,}")

    def clean_text(self, text: str) -> str:
        # Step 1: Remove systemic boilerplate and targeted unneeded sections
        text = self.BOILERPLATE_PATTERN.sub("", text)
        text = self.NAVIGATION_SECTION_PATTERN.sub("", text)
        text = self.MERMAID_PATTERN.sub("", text)
        
        # Step 2: Inline syntax cleanup (Preserves lines & keeps table strings inline)
        text = self.IMAGE_PATTERN.sub(r"\1", text)
        text = self.LINK_PATTERN.sub(r"\1", text)
        text = self.DEFINITION_LIST_PATTERN.sub(r"\1: \2", text)
        text = self.BLOCKQUOTE_PATTERN.sub("", text)
        
        # Step 3: Structural whitespace normalization
        text = self.EXCESS_NEWLINES_PATTERN.sub("\n\n", text)
        return text.strip()

    def clean_documents(self, documents: List[Document]) -> List[Document]:
        cleaned_documents = []
        for document in documents:
            cleaned_content = self.clean_text(document.page_content)
            cleaned_documents.append(
                Document(
                    page_content=cleaned_content,
                    metadata=document.metadata
                )
            )
        return cleaned_documents

In [43]:
cleaner = MarkdownCleaner()
cleaned_docs = cleaner.clean_documents(markdown_docs)

In [44]:
cleaned_docs

[Document(page_content='# NemoClaw Architecture Overview\nLearn how NemoClaw combines a host CLI, sandbox plugin, and versioned blueprint to move OpenClaw into a controlled sandbox.\n\nThis page explains how NemoClaw runs OpenClaw inside an OpenShell sandbox and how the gateway connects the agent to inference, integrations, and policy.\n\nNemoClaw does not replace OpenClaw or OpenShell.\nIt packages them into a repeatable setup with a host CLI, a versioned blueprint, default policies, inference setup, plugin configuration, and state helpers.\nYou can use that setup directly or adapt it for your own OpenShell integration.\n\n## High-Level Flow\n\nNemoClaw keeps the user workflow on the host while OpenShell enforces the sandbox boundary.\nThe gateway sits between NemoClaw control, the sandbox, inference providers, and external integrations.\nThat placement lets NemoClaw configure the environment without giving the agent direct access to host credentials or uncontrolled network egress.\n\

In [45]:
with open("cleaner_outputs.txt", "w", encoding="utf-8") as f:
    for doc in cleaned_docs: 
        f.write(f"{doc}\n")

# Updated Cleaner

In [7]:
import re
from typing import List
from src.models.document import Document

class MarkdownCleaner1:
    def __init__(self):
        # 1. Matches exactly the Fern boilerplate at the start of files
        self.BOILERPLATE_PATTERN = re.compile(
            r"^>\s*For clean Markdown.*\n"
            r"^>\s*For a complete documentation index.*\n"
            r"^>\s*For full documentation content.*\n"
            r"^>\s*For AI client integration.*\n",
            flags=re.MULTILINE
        )

        # 2. Removes specific navigation/footer sections cleanly without over-greedy matching
        self.NAVIGATION_SECTION_PATTERN = re.compile(
            r"^##\s+(?:Next Steps|Related topics)\s*\n(?:(?!^##\s+).)*",
            flags=re.MULTILINE | re.DOTALL
        )

        # 3. Removes Mermaid code blocks cleanly
        self.MERMAID_PATTERN = re.compile(
            r"```mermaid\n.*?```\n?",
            flags=re.DOTALL
        )

        # 4. Extract caption from image tag: ![Caption](url) -> Caption
        self.IMAGE_PATTERN = re.compile(r"!\[([^\]]*)]\([^\)]+\)")

        # 5. Extract anchor text from markdown links: [Anchor](url) -> Anchor
        self.LINK_PATTERN = re.compile(r"\[([^\]]+)]\([^\)]+\)")

        # 6. Normalizes definition lists
        self.DEFINITION_LIST_PATTERN = re.compile(
            r"^([^\n]+)\n:\s*(.+)$",
            flags=re.MULTILINE
        )

        # 7. Removes leading blockquote markers
        self.BLOCKQUOTE_PATTERN = re.compile(r"^\s*>\s*", flags=re.MULTILINE)

        # 8. Cleans excess vertical whitespace
        self.EXCESS_NEWLINES_PATTERN = re.compile(r"\n{3,}")

    def repair_table_line_breaks(self, text: str) -> str:
        """
        Ensures that markdown table structures fragmented across newlines 
        are re-assembled into continuous lines for proper markdown splitting.
        """
        lines = text.splitlines()
        repaired_lines = []
        in_table_zone = False

        for line in lines:
            stripped = line.strip()
            # Detect table rows or markdown table dividers
            if stripped.startswith('|') or (in_table_zone and stripped.endswith('|')):
                # Check if it's a divider row or complete row closure
                if stripped.startswith('|') and stripped.endswith('|') and len(repaired_lines) > 0 and repaired_lines[-1].strip().endswith('|'):
                    # If the previous line didn't close its row or if this line continues a row fragment
                    if '---' in stripped:
                        in_table_zone = True
                        repaired_lines.append(line)
                    else:
                        # Append directly or fix dangling layout
                        repaired_lines.append(line)
                elif in_table_zone and not repaired_lines[-1].strip().endswith('|'):
                    # Join the line fragment to clean up internal string splits
                    repaired_lines[-1] = repaired_lines[-1].rstrip() + " " + stripped
                else:
                    repaired_lines.append(line)
                in_table_zone = True
            else:
                repaired_lines.append(line)
                in_table_zone = False

        return "\n".join(repaired_lines)

    def clean_text(self, text: str) -> str:
        # Block step adjustments
        text = self.BOILERPLATE_PATTERN.sub("", text)
        text = self.NAVIGATION_SECTION_PATTERN.sub("", text)
        text = self.MERMAID_PATTERN.sub("", text)
        
        # Inline syntax conversions
        text = self.IMAGE_PATTERN.sub(r"\1", text)
        text = self.LINK_PATTERN.sub(r"\1", text)
        text = self.DEFINITION_LIST_PATTERN.sub(r"\1: \2", text)
        text = self.BLOCKQUOTE_PATTERN.sub("", text)
        
        # Table protection line assembly pass
        text = self.repair_table_line_breaks(text)
        
        # Structural whitespace normalization
        text = self.EXCESS_NEWLINES_PATTERN.sub("\n\n", text)
        return text.strip()

    def clean_documents(self, documents: List[Document]) -> List[Document]:
        cleaned_documents = []
        for document in documents:
            cleaned_content = self.clean_text(document.page_content)
            cleaned_documents.append(
                Document(
                    page_content=cleaned_content,
                    metadata=document.metadata
                )
            )
        return cleaned_documents

In [8]:
cleaner_updated = MarkdownCleaner1()
cleaned_md_docs = cleaner_updated.clean_documents(markdown_docs)

In [9]:
cleaned_md_docs

[Document(page_content='# NemoClaw Architecture Overview\nLearn how NemoClaw combines a host CLI, sandbox plugin, and versioned blueprint to move OpenClaw into a controlled sandbox.\n\nThis page explains how NemoClaw runs OpenClaw inside an OpenShell sandbox and how the gateway connects the agent to inference, integrations, and policy.\n\nNemoClaw does not replace OpenClaw or OpenShell.\nIt packages them into a repeatable setup with a host CLI, a versioned blueprint, default policies, inference setup, plugin configuration, and state helpers.\nYou can use that setup directly or adapt it for your own OpenShell integration.\n\n## High-Level Flow\n\nNemoClaw keeps the user workflow on the host while OpenShell enforces the sandbox boundary.\nThe gateway sits between NemoClaw control, the sandbox, inference providers, and external integrations.\nThat placement lets NemoClaw configure the environment without giving the agent direct access to host credentials or uncontrolled network egress.\n\

In [51]:
with open("output.txt", "w", encoding="utf-8") as f:
    for doc in cleaned_md_docs: 
        f.write(f"{doc}\n")

# Chunker.py

In [52]:
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from src.models.document import Document, DocumentMetadata

# =========================================================
# SPLITTER CONFIGURATION
# =========================================================

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
    ], 
    strip_headers=False 
)

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def extract_section_name(metadata: dict) -> str | None:
    """Extract the most specific available markdown header."""
    return (
        metadata.get("header_3")
        or metadata.get("header_2")
        or metadata.get("header_1")
    )


def generate_chunk_id(source: str, chunk_index: int, page: int | None = None) -> str:
    """Generate deterministic, globally unique readable chunk IDs."""
    stem = Path(source).stem.lower().replace(" ", "_")
    if page is not None:
        return f"{stem}_page_{page}_chunk_{chunk_index}"
    return f"{stem}_chunk_{chunk_index}"


def create_chunk_document(
    content: str,
    original_metadata: DocumentMetadata,
    chunk_index: int,
    section: str | None = None,
) -> Document:
    """Create a chunked Document object with enriched metadata."""
    chunk_id = generate_chunk_id(
        source=original_metadata.source,
        page=original_metadata.page,
        chunk_index=chunk_index,
    )

    parent_document_id = Path(original_metadata.source).stem.lower().replace(" ", "_")

    metadata = DocumentMetadata(
        source=original_metadata.source,
        file_type=original_metadata.file_type,
        page=original_metadata.page,
        section=section,
        chunk_id=chunk_id,
        chunk_index=chunk_index,
        parent_document_id=parent_document_id,
    )

    return Document(page_content=content, metadata=metadata)

# =========================================================
# FIXED MAIN DISPATCHER & STRUCTURAL LOOPS
# =========================================================

def chunk_documents_new(documents: List[Document]) -> List[Document]:
    """
    Main chunking dispatcher. Keeps global counters and carries over context state 
    to fix repetitive indexing and null sections across pages.
    """
    all_chunks = []
    
    # FIX 1: Track separate global counters per unique document file
    document_counters = {}
    
    # FIX 2: Track the last running active section layout across page sets
    last_active_section = None

    for document in documents:
        file_type = document.metadata.file_type.lower()
        source_file = document.metadata.source
        
        # Initialize unique file index trackers dynamically
        if source_file not in document_counters:
            document_counters[source_file] = 0

        # --- PROCESS PDF DOCUMENTS ---
        if file_type in ["pdf"]:
            header_splits = markdown_splitter.split_text(document.page_content)

            for split in header_splits:
                extracted_section = extract_section_name(split.metadata)
                
                # Context Carry-Over Strategy: 
                # If LangChain didn't parse a header on this structural slice,
                # use the ongoing last known parent section layout.
                if extracted_section:
                    last_active_section = extracted_section
                
                # Split content into chunk text strings safely
                recursive_chunks = recursive_splitter.split_text(split.page_content)

                for chunk_text in recursive_chunks:
                    chunk_doc = create_chunk_document(
                        content=chunk_text,
                        original_metadata=document.metadata,
                        chunk_index=document_counters[source_file], # Using continuous counter
                        section=last_active_section,               # Restored parent section
                    )
                    all_chunks.append(chunk_doc)
                    document_counters[source_file] += 1

        # --- PROCESS MD CHUNKS ---
        elif file_type in ["md", "markdown"]:
            header_splits = markdown_splitter.split_text(document.page_content)

            for split in header_splits:
                extracted_section = extract_section_name(split.metadata)

                chunk_doc = create_chunk_document(
                    content=split.page_content,
                    original_metadata=document.metadata,
                    chunk_index=document_counters[source_file],
                    section=extracted_section,
                )

                all_chunks.append(chunk_doc)
                document_counters[source_file] += 1

        # --- PROCESS PLAIN TEXT CHUNKS ---
        elif file_type == "txt":
            chunks = recursive_splitter.split_text(document.page_content)

            for chunk_text in chunks:
                chunk_doc = create_chunk_document(
                    content=chunk_text,
                    original_metadata=document.metadata,
                    chunk_index=document_counters[source_file],
                    section=None,
                )
                all_chunks.append(chunk_doc)
                document_counters[source_file] += 1

        else:
            raise ValueError(f"Unsupported file type: {file_type}")

    return all_chunks

In [53]:
markdown_chunks = chunk_documents_new(cleaned_md_docs)

In [54]:
markdown_chunks?

Type:        list
String form: [Document(page_content='# NemoClaw Architecture Overview\nLearn how NemoClaw combines a host CLI, <...> d='nemoclaw_release_notes_chunk_4', chunk_index=4, parent_document_id='nemoclaw_release_notes'))]
Length:      30
Docstring:  
Built-in mutable sequence.

If no argument is given, the constructor creates a new empty list.
The argument must be an iterable if specified.

In [55]:
markdown_chunks[0]

Document(page_content='# NemoClaw Architecture Overview\nLearn how NemoClaw combines a host CLI, sandbox plugin, and versioned blueprint to move OpenClaw into a controlled sandbox.  \nThis page explains how NemoClaw runs OpenClaw inside an OpenShell sandbox and how the gateway connects the agent to inference, integrations, and policy.  \nNemoClaw does not replace OpenClaw or OpenShell.\nIt packages them into a repeatable setup with a host CLI, a versioned blueprint, default policies, inference setup, plugin configuration, and state helpers.\nYou can use that setup directly or adapt it for your own OpenShell integration.', metadata=DocumentMetadata(source='NemoClaw Architecture Overview.md', file_type='md', page=None, section='NemoClaw Architecture Overview', chunk_id='nemoclaw_architecture_overview_chunk_0', chunk_index=0, parent_document_id='nemoclaw_architecture_overview'))

In [56]:
markdown_chunks

[Document(page_content='# NemoClaw Architecture Overview\nLearn how NemoClaw combines a host CLI, sandbox plugin, and versioned blueprint to move OpenClaw into a controlled sandbox.  \nThis page explains how NemoClaw runs OpenClaw inside an OpenShell sandbox and how the gateway connects the agent to inference, integrations, and policy.  \nNemoClaw does not replace OpenClaw or OpenShell.\nIt packages them into a repeatable setup with a host CLI, a versioned blueprint, default policies, inference setup, plugin configuration, and state helpers.\nYou can use that setup directly or adapt it for your own OpenShell integration.', metadata=DocumentMetadata(source='NemoClaw Architecture Overview.md', file_type='md', page=None, section='NemoClaw Architecture Overview', chunk_id='nemoclaw_architecture_overview_chunk_0', chunk_index=0, parent_document_id='nemoclaw_architecture_overview')),
 Document(page_content='## High-Level Flow  \nNemoClaw keeps the user workflow on the host while OpenShell en

In [57]:
with open("chunk_output.txt", "w", encoding="utf-8") as f:
    for chunk in markdown_chunks: 
        f.write(f"{chunk}\n")

# New chunker

In [10]:
from pathlib import Path
from typing import List

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
from src.models.document import Document, DocumentMetadata

# =========================================================
# SPLITTER CONFIGURATION
# =========================================================

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100, separators=["\n\n", "\n", ". ", " ", ""]
)

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
    ],
    strip_headers=False,  # Crucial for preserving structural layout context
)

# =========================================================
# HELPER FUNCTIONS
# =========================================================


def extract_section_name(metadata: dict) -> str | None:
    """Extract the most specific available markdown header."""
    return (
        metadata.get("header_3") or metadata.get("header_2") or metadata.get("header_1")
    )


def generate_chunk_id(source: str, chunk_index: int, page: int | None = None) -> str:
    """Generate deterministic, globally unique readable chunk IDs."""
    stem = Path(source).stem.lower().replace(" ", "_")
    if page is not None:
        return f"{stem}_page_{page}_chunk_{chunk_index}"
    return f"{stem}_chunk_{chunk_index}"


def create_chunk_document(
    content: str,
    original_metadata: DocumentMetadata,
    chunk_index: int,
    section: str | None = None,
) -> Document:
    """Create a chunked Document object with enriched metadata."""
    chunk_id = generate_chunk_id(
        source=original_metadata.source,
        page=original_metadata.page,
        chunk_index=chunk_index,
    )

    parent_document_id = Path(original_metadata.source).stem.lower().replace(" ", "_")

    metadata = DocumentMetadata(
        source=original_metadata.source,
        file_type=original_metadata.file_type,
        page=original_metadata.page,
        section=section,
        chunk_id=chunk_id,
        chunk_index=chunk_index,
        parent_document_id=parent_document_id,
    )

    return Document(page_content=content, metadata=metadata)


# =========================================================
# FIXED MAIN DISPATCHER & STRUCTURAL LOOPS
# =========================================================


def chunk_documents_new(documents: List[Document]) -> List[Document]:
    """
    Main chunking dispatcher with size constraints enforced across all document routes.
    """
    all_chunks = []
    document_counters = {}

    for document in documents:
        file_type = document.metadata.file_type.lower()
        source_file = document.metadata.source

        if source_file not in document_counters:
            document_counters[source_file] = 0

        # --- PROCESS PDF AND MD DOCUMENTS (Hybrid Strategy) ---
        if file_type in ["pdf", "md", "markdown"]:
            header_splits = markdown_splitter.split_text(document.page_content)
            last_active_section = None

            for split in header_splits:
                extracted_section = extract_section_name(split.metadata)

                if extracted_section:
                    last_active_section = extracted_section

                # Enforce chunk_size constraints via Recursive Character Splitting
                recursive_chunks = recursive_splitter.split_text(split.page_content)

                for chunk_text in recursive_chunks:
                    chunk_doc = create_chunk_document(
                        content=chunk_text,
                        original_metadata=document.metadata,
                        chunk_index=document_counters[source_file],
                        section=last_active_section,
                    )
                    all_chunks.append(chunk_doc)
                    document_counters[source_file] += 1

        # --- PROCESS PLAIN TEXT CHUNKS ---
        elif file_type == "txt":
            chunks = recursive_splitter.split_text(document.page_content)

            for chunk_text in chunks:
                chunk_doc = create_chunk_document(
                    content=chunk_text,
                    original_metadata=document.metadata,
                    chunk_index=document_counters[source_file],
                    section=None,
                )
                all_chunks.append(chunk_doc)
                document_counters[source_file] += 1

        else:
            raise ValueError(f"Unsupported file type: {file_type}")

    return all_chunks


In [12]:
markdown_chunks = chunk_documents_new(cleaned_md_docs)

In [13]:
markdown_chunks?

Type:        list
String form: [Document(page_content='# NemoClaw Architecture Overview\nLearn how NemoClaw combines a host CLI, <...> 'nemoclaw_release_notes_chunk_26', chunk_index=26, parent_document_id='nemoclaw_release_notes'))]
Length:      182
Docstring:  
Built-in mutable sequence.

If no argument is given, the constructor creates a new empty list.
The argument must be an iterable if specified.

In [14]:
with open("chunk_outputs.txt", "w", encoding="utf-8") as f:
    for chunk in markdown_chunks: 
        f.write(f"{chunk}\n")